# Неделя 3 — Признаки и CatBoost

Задание: `docs/week3_baseline.md`

In [9]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

dataset = pd.read_parquet('../data/processed/dataset.parquet')
CAT_COLS = ['segment', 'product', 'region']
NUM_COLS = [c for c in dataset.columns
            if c not in CAT_COLS + ['client_id', 'snapshot_date', 'target']]
print(len(NUM_COLS), 'числовых признаков')

fit_df = dataset[dataset['snapshot_date'].isin(['2024-07', '2024-10', '2025-01'])]
val_df = dataset[dataset['snapshot_date'] == '2025-04']
test_df = dataset[dataset['snapshot_date'].isin(['2025-07', '2025-10'])]

X_fit, y_fit = fit_df[NUM_COLS + CAT_COLS], fit_df['target']
X_val, y_val = val_df[NUM_COLS + CAT_COLS], val_df['target']
X_te, y_te = test_df[NUM_COLS + CAT_COLS], test_df['target']

18 числовых признаков


## 1. Построение признаков по группам

TODO: реализуйте make_features() в src/features.py (платежи, динамика, объём отношений, сигналы боли).

## 2. Обучение CatBoost

TODO: CatBoostClassifier с early stopping на eval-срезе по времени (не случайном!).

## 3. Итоговая таблица метрик: правило vs логрегрессия vs CatBoost

In [10]:
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Baseline 0: правило
m_rule = evaluate(y_te, -X_te['revenue_last_to_mean'], 'Baseline 0: правило')

# Baseline 1: логрегрессия
def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

preprocess = ColumnTransformer([
    ('num', Pipeline([('log', FunctionTransformer(signed_log1p, validate=False)),
                      ('scale', RobustScaler())]), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
])
model_lr = Pipeline([('pre', preprocess),
                     ('clf', LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced'))])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    model_lr.fit(X_fit, y_fit)
score_lr = model_lr.predict_proba(X_te)[:, 1]
m_lr = evaluate(y_te, score_lr, 'Baseline 1: логрегрессия')

# Baseline 2: CatBoost с ранней остановкой на val
model_cb = CatBoostClassifier(
    iterations=1000,
    early_stopping_rounds=100,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=False,
)
model_cb.fit(X_fit, y_fit, cat_features=CAT_COLS,
             eval_set=(X_val, y_val), use_best_model=True)
score_cb = model_cb.predict_proba(X_te)[:, 1]
m_cb = evaluate(y_te, score_cb, 'Baseline 2: CatBoost')
print('best_iteration:', model_cb.best_iteration_)

# Итоговая таблица
final = pd.DataFrame([m_rule, m_lr, m_cb]).set_index('label')
print(final)

--- Baseline 0: правило ---
ROC-AUC: 0.9067
PR-AUC: 0.4314
Precision@10%: 0.4204
Lift@10%: 8.25x
Доля оттока: 0.0510
--- Baseline 1: логрегрессия ---
ROC-AUC: 0.9246
PR-AUC: 0.5901
Precision@10%: 0.4306
Lift@10%: 8.44x
Доля оттока: 0.0510
--- Baseline 2: CatBoost ---
ROC-AUC: 0.9461
PR-AUC: 0.6540
Precision@10%: 0.4452
Lift@10%: 8.73x
Доля оттока: 0.0510
best_iteration: 74
                           roc_auc    pr_auc  precision_at_10  lift_at_10  \
label                                                                       
Baseline 0: правило       0.906709  0.431374         0.420422    8.245640   
Baseline 1: логрегрессия  0.924568  0.590054         0.430581    8.444884   
Baseline 2: CatBoost      0.946120  0.654017         0.445168    8.730978   

                          churn_rate  
label                                 
Baseline 0: правило         0.050987  
Baseline 1: логрегрессия    0.050987  
Baseline 2: CatBoost        0.050987  


## 4. Вывод на языке бизнеса

TODO: одна фраза вида 'в топ-10% клиентов по риску попадает X% всех будущих уходов, lift = Yx'.

In [11]:
lift = m_cb['lift_at_10']
capture = lift * 0.10
print(f"В топ-10% клиентов по риску попадает {capture:.0%} всех будущих уходов, "
      f"lift = {lift:.1f}x (ROC-AUC = {m_cb['roc_auc']:.3f})")

В топ-10% клиентов по риску попадает 87% всех будущих уходов, lift = 8.7x (ROC-AUC = 0.946)
